In [ ]:
import subprocess, sys, os, time
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate bitsandbytes 2>&1 | tail -3")
sh(f'{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1')

In [ ]:
# Chấm UGround các câu chạm mới của lượt tuần tự (kernel tuantu-lichsu-tusinh, khâu chấm lỗi vì chốt tệp thiếu).
# --n 99999: chốt của score_run chỉ cho chấm tệp có một phần bước chạm khi có --n; n lớn hơn số bước nên không cắt bước nào.
import os, glob, json, shutil, threading
W = "/kaggle/working"; OUT = f"{W}/out"; os.makedirs(OUT, exist_ok=True)
T0 = time.time(); TOM = open(f"{OUT}/tom_tat.txt", "a")
def ghi(*a):
    s = f"[{(time.time()-T0)/3600:5.2f} h] " + " ".join(map(str, a)); print(s, flush=True); TOM.write(s + "\n"); TOM.flush()
import torch
assert torch.cuda.device_count() == 2, "DỪNG: cần 2 GPU"
KIT = os.path.dirname(glob.glob("/kaggle/input/**/gen_tuantu.py", recursive=True)[0])
AC = f"{KIT}/ac"
os.makedirs(f"{W}/harness", exist_ok=True)
for f in glob.glob(f"{KIT}/harness/*.py"): shutil.copy(f, f"{W}/harness/")
PM = {nh: glob.glob(f"/kaggle/input/**/preds_moi_{nh}.jsonl", recursive=True) for nh in ("s1", "ck500")}
ghi("tệp câu mới:", PM)
N = {"s1": 2672, "ck500": 2603}
for nh in N:
    assert len(PM[nh]) == 1 and sum(1 for _ in open(PM[nh][0])) == N[nh], f"DỪNG: {nh}"
def cham(nh, gpu):
    log = f"{OUT}/cham_{nh}.log"
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}
    with open(log, "a") as f:
        q = subprocess.Popen(["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--data-root", AC,
                              "--preds", PM[nh][0], "--n", "99999", "--out", f"{OUT}/score_moi_{nh}.json"],
                             cwd=W, stdout=f, stderr=subprocess.STDOUT, env=env, start_new_session=True)
        while q.poll() is None:
            time.sleep(120)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"  {(time.time()-T0)/3600:5.2f} h · {nh} · {L[-1][:100] if L else ''}", flush=True)
    raw = f"{OUT}/score_moi_{nh}_raw.jsonl"
    ghi(nh, "mã thoát", q.returncode, "· dòng raw", sum(1 for _ in open(raw)) if os.path.exists(raw) else 0)
ts = [threading.Thread(target=cham, args=(nh, g)) for nh, g in (("s1", 0), ("ck500", 1))]
for t in ts: t.start()
for t in ts: t.join()

In [ ]:
sh(f"cd {W} && zip -qr cham_out.zip out && ls -la {W}")
ghi("KẾT THÚC")